In [31]:
import numpy as np
import torch
import itertools as itr
from scipy.special import softmax
from torch import nn
from common.ffn.ffn_relu import ParametricReLUNet
#from common.coeff_calc.coeff_calc_nlo import MCSimulator
from common.coeff_calc.coeff_calc_ntk import NTKSimulator

from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

In [10]:
# use it to conver from PIL to torch.Tensor
image_transform = ToTensor()

train_dataset = MNIST(root='./', train=True, download=True, transform=image_transform)
test_dataset = MNIST(root='./', train=False, download=True, transform=image_transform)

In [11]:
BATCH_SIZE = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

In [12]:
class MNISTReLU(ParametricReLUNet):
    def __init__(self, input_dim, output_dim):
        super().__init__(n0=input_dim,nk=0,nl=output_dim,l=0, bias_on=True)
                
        self.input_fc = nn.Linear(input_dim, 250)
        self.hidden_fc = nn.Linear(250, 100)
        self.output_fc = nn.Linear(100, output_dim)

    def forward_(self, x):
        return self.forward(x)

    def forward(self, x):
        #x = [batch size, height, width]
        #batch_size = x.shape[0]
        #x = x.view(batch_size, -1)
        #x = [batch size, height * width]
        h_1 = self.PReLU(self.input_fc(x))
        #h_1 = [batch size, 250]
        h_2 = self.PReLU(self.hidden_fc(h_1))
        #h_2 = [batch size, 100]
        y_pred = self.output_fc(h_2)
        #y_pred = [batch size, output dim]
        return y_pred
    
    def init_weights(self, cb=0.0, cw=1.0):
        if self.get_log_level() == "debug":
            print("FeedForwardNet weights initialisation with cb={}, cw={}".format(cb, cw))

        #Weight initialisation as in 2.19, 2.20
        self.cb, self.cw = cb, cw
        self.init_linear_weights(self.input_fc, self.bias_on, cb, cw/self.input_fc.in_features)
        self.init_linear_weights(self.hidden_fc, self.bias_on, cb, cw/self.hidden_fc.in_features)
        self.init_linear_weights(self.output_fc, self.bias_on, cb, cw/self.output_fc.in_features)


In [13]:
INPUT_DIM = 28 * 28
OUTPUT_DIM = 10  # num classes

#Weights distribution variances are set as in (5.67)
slope_plus, slope_minus=1.0, 0.0
cb, cw = 0, 2.0/(slope_plus**2.0 + slope_minus**2.0)

testNet = MNISTReLU(INPUT_DIM, OUTPUT_DIM)
testNet.set_log_level("info")
testNet.set_slopes(slope_plus, slope_minus)
testNet.init_weights(cb, cw)


#Loss function - extended cross-enthropy

FeedForwardNet created with n0=784, nk=0, nl=10, l=0, bias_on=True


In [47]:

def PReLUz(input: float) -> float:
    return slope_plus * input if input >= 0 else slope_minus * input

def PReLUdrv(input: float) -> float:
    return slope_plus if input >= 0 else slope_minus

act = np.vectorize(PReLUz)
drv = np.vectorize(PReLUdrv)

ln=3
lb, lw, eta = 1,1,1 #1e-2, 1e-2, 1

images, labels = next(iter(train_dataloader))
batch_size = images.shape[0]
xx = images.view(batch_size, -1)
#print(xx.shape)
logits = testNet.forward_(xx)

#Calculate NTK
KK = np.zeros((ln, BATCH_SIZE, BATCH_SIZE))
Theta = np.zeros_like(KK)

sim = NTKSimulator(cb=cb, cw=cw, lb=lb, lw=lw, act=act, drv=drv, n_samples=100000)
sim.calculate_layer0(KK, Theta, np.transpose(xx.detach().numpy()))
for idx in range(1, ln):
    sim.calculate_layer(KK, Theta, idx)
    print('-', end='')


--

In [71]:
#print(logits[0])



input_fc_weight = np.zeros((BATCH_SIZE, OUTPUT_DIM, 250, 784))
input_fc_bias = np.zeros((BATCH_SIZE, OUTPUT_DIM, 250))
hidden_fc_weight = np.zeros((BATCH_SIZE, OUTPUT_DIM, 100, 250))
hidden_fc_bias = np.zeros((BATCH_SIZE, OUTPUT_DIM, 100))
output_fc_weight = np.zeros((BATCH_SIZE, OUTPUT_DIM, 10, 100))
output_fc_bias = np.zeros((BATCH_SIZE, OUTPUT_DIM, 10))

for alpha, kk in itr.product(range(BATCH_SIZE),range(OUTPUT_DIM)):
    df = torch.autograd.grad(logits[alpha,kk], (testNet.input_fc.weight, testNet.input_fc.bias\
                                    , testNet.hidden_fc.weight, testNet.hidden_fc.bias\
                                    , testNet.output_fc.weight, testNet.output_fc.bias)\
        , retain_graph=True, create_graph=True, allow_unused=True)    
    input_fc_weight[alpha, kk] = df[0].detach().numpy()
    input_fc_bias[alpha, kk] = df[1].detach().numpy()
    hidden_fc_weight[alpha, kk] = df[2].detach().numpy()
    hidden_fc_bias[alpha, kk] = df[3].detach().numpy()
    output_fc_weight[alpha, kk] = df[4].detach().numpy()
    output_fc_bias[alpha, kk] = df[4].detach().numpy()


In [84]:
print(logits.shape, df[0].shape, df[1].shape, df[2].shape, df[3].shape, df[4].shape, df[5].shape)
#print(testNet.input_fc.weight.grad)

torch.Size([128, 10]) torch.Size([250, 784]) torch.Size([250]) torch.Size([100, 250]) torch.Size([100]) torch.Size([10, 100]) torch.Size([10])


In [19]:
thetal = Theta[-1]
ktop = np.linalg.inv(thetal)
ktop
#dz^(L)_(k;α1)/dθν

array([[ 258.87369749,   24.14420971,   19.40620604, ...,  -53.83274627,
          -1.49766858,    8.37855318],
       [  24.14420971,  497.88688465,  -32.44847752, ...,  -36.17224513,
         -40.60211645,  -12.61035554],
       [  19.40620604,  -32.44847752,  220.84651279, ...,    9.19559844,
         -10.54282816,   63.80798422],
       ...,
       [ -53.83274627,  -36.17224513,    9.19559844, ..., 1318.04562075,
         -77.04351088,   -8.17221477],
       [  -1.49766858,  -40.60211645,  -10.54282816, ...,  -77.04351088,
         377.49278866, -149.58685931],
       [   8.37855318,  -12.61035554,   63.80798422, ...,   -8.17221477,
        -149.58685931,  471.7834645 ]])

In [16]:
'''
#Softmax for prediction q and labels p
qq = softmax(logits.detach().numpy(), axis=1)
print(qq.shape)
pp = np.full((BATCH_SIZE, OUTPUT_DIM), 0.08533674)
for pos in range(BATCH_SIZE):
    pp[pos, labels[pos]] = 0.23196932

vary = (pp - qq).flatten()

thetal = Theta[-1]
one = np.eye(OUTPUT_DIM, dtype=np.float64)
VARX = np.zeros((BATCH_SIZE*OUTPUT_DIM, BATCH_SIZE*BATCH_SIZE), np.float64)
batch_range = range(BATCH_SIZE)

#for alpha in range(BATCH_SIZE):
#    for beta in range(BATCH_SIZE):
for alpha, beta in itr.product(batch_range,batch_range):
    #print(alpha, beta)
    col = alpha*BATCH_SIZE+beta
    pbeta_qalpha = pp[beta] - qq[alpha]
    for delta in batch_range:
        delta_term = eta*thetal[delta,alpha]
        for nn in range(OUTPUT_DIM):
            row = delta*OUTPUT_DIM+nn
#for row in range(BATCH_SIZE*OUTPUT_DIM):
    #delta, nn = row // OUTPUT_DIM, row % OUTPUT_DIM
            #alpha, beta = col // BATCH_SIZE, row % BATCH_SIZE
            #sum(np.multiply(pbeta_qalpha, one[nn]-qq[delta]))
            #term = #sum((pbeta_qalpha[mm])*((1 if mm == nn else 0) - qq[delta, mm]) for mm in range(OUTPUT_DIM))
            #for mm in range(OUTPUT_DIM):
            #    term += (pp[beta, mm] - qq[alpha, mm])*(1 if mm == nn else 0 - qq[delta, mm])
            VARX[row, col] = delta_term*qq[delta, nn]*sum(np.multiply(pbeta_qalpha, one[nn]-qq[delta]))
            #sum((pp[beta, mm] - qq[alpha, mm])*((1 if mm == nn else 0) - qq[delta, mm]) for mm in range(OUTPUT_DIM))

KAB_aggr = np.linalg.lstsq(VARX, vary, rcond=None)
'''

'\n#Softmax for prediction q and labels p\nqq = softmax(logits.detach().numpy(), axis=1)\nprint(qq.shape)\npp = np.full((BATCH_SIZE, OUTPUT_DIM), 0.08533674)\nfor pos in range(BATCH_SIZE):\n    pp[pos, labels[pos]] = 0.23196932\n\nvary = (pp - qq).flatten()\n\nthetal = Theta[-1]\none = np.eye(OUTPUT_DIM, dtype=np.float64)\nVARX = np.zeros((BATCH_SIZE*OUTPUT_DIM, BATCH_SIZE*BATCH_SIZE), np.float64)\nbatch_range = range(BATCH_SIZE)\n\n#for alpha in range(BATCH_SIZE):\n#    for beta in range(BATCH_SIZE):\nfor alpha, beta in itr.product(batch_range,batch_range):\n    #print(alpha, beta)\n    col = alpha*BATCH_SIZE+beta\n    pbeta_qalpha = pp[beta] - qq[alpha]\n    for delta in batch_range:\n        delta_term = eta*thetal[delta,alpha]\n        for nn in range(OUTPUT_DIM):\n            row = delta*OUTPUT_DIM+nn\n#for row in range(BATCH_SIZE*OUTPUT_DIM):\n    #delta, nn = row // OUTPUT_DIM, row % OUTPUT_DIM\n            #alpha, beta = col // BATCH_SIZE, row % BATCH_SIZE\n            #sum(np.